In [2]:
import pandas as pd
import numpy as np
from datetime import datetime

In [5]:
customers = pd.read_csv(r'C:\Users\Ben Ten\OneDrive\Desktop\Projects\Final\Marketing\data\raw data\raw_customers.csv')
orders = pd.read_csv(r'C:\Users\Ben Ten\OneDrive\Desktop\Projects\Final\Marketing\data\raw data\raw_orders.csv')
responses = pd.read_csv(r'C:\Users\Ben Ten\OneDrive\Desktop\Projects\Final\Marketing\data\raw data\raw_campaign_responses.csv')

In [6]:
# Cleaning and preprocessing

# Orders:
print('Orders Data Overview:')
print('------------------------------------')
print(f'Orders Shape: {orders.shape}')
print('------------------------------------')
print(f'Orders Info:')
print(orders.info())
print('------------------------------------')
print(f'Orders Describe:')
print(orders.describe())
print('------------------------------------')
print(f'Orders Null Values: {orders.isnull().sum()}')
print('------------------------------------')
print(f'Orders Duplicates: {orders.duplicated().sum()}')
print('------------------------------------')
print(f'Orders Negative Values: {(orders['amount'] < 0).sum()}')

Orders Data Overview:
------------------------------------
Orders Shape: (20200, 7)
------------------------------------
Orders Info:
<class 'pandas.DataFrame'>
RangeIndex: 20200 entries, 0 to 20199
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     20200 non-null  int64  
 1   customer_id  20200 non-null  int64  
 2   order_date   20200 non-null  str    
 3   amount       20150 non-null  float64
 4   category     20200 non-null  str    
 5   channel      20200 non-null  str    
 6   status       20200 non-null  str    
dtypes: float64(1), int64(2), str(4)
memory usage: 1.1 MB
None
------------------------------------
Orders Describe:
           order_id   customer_id        amount
count  20200.000000  20200.000000  20150.000000
mean   14996.706436   3503.619356    333.982054
std     5775.182333   1446.419400    322.823072
min     5001.000000   1002.000000  -1769.570000
25%     9994.750000   2244.750000 

In [12]:
# Customers:
print('Customers Data Overview:')
print('------------------------------------')
print(f'Customers Shape: {customers.shape}')
print('------------------------------------')
print(f'Customers Info:')
print(customers.info())
print('------------------------------------')
print(f'Customers Describe:')
print(customers.describe())
print('------------------------------------')
print(f'Customers Null Values: {customers.isnull().sum()}')
print('------------------------------------')
print(f'Customers Duplicates: {customers.duplicated().sum()}')
print('------------------------------------')
print(f'Customers Age Negative Values: {(customers["age"] < 0).sum()}')
print('------------------------------------')
print(f'Customer Sign-up Dates: ({customers["signup_date"].min()}, {customers["signup_date"].max()})')

Customers Data Overview:
------------------------------------
Customers Shape: (5000, 8)
------------------------------------
Customers Info:
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   customer_id     5000 non-null   int64
 1   name            5000 non-null   str  
 2   email           5000 non-null   str  
 3   city            5000 non-null   str  
 4   signup_channel  5000 non-null   str  
 5   signup_date     5000 non-null   str  
 6   age             5000 non-null   int64
 7   gender          5000 non-null   str  
dtypes: int64(2), str(6)
memory usage: 312.6 KB
None
------------------------------------
Customers Describe:
       customer_id          age
count  5000.000000  5000.000000
mean   3500.500000    40.475000
std    1443.520003    13.471335
min    1001.000000    18.000000
25%    2250.750000    29.000000
50%    3500.500000    40.000000

In [9]:
# Responses:
print('Responses Data Overview:')
print('------------------------------------')
print(f'Responses Shape: {responses.shape}')
print('------------------------------------')
print(f'Responses Info:')
print(responses.info())
print('------------------------------------')
print(f'Responses Describe:')
print(responses.describe())
print('------------------------------------')
print(f'Responses Null Values: {responses.isnull().sum()}')
print('------------------------------------')
print(f'Responses Duplicates: {responses.duplicated().sum()}')
print('------------------------------------')
print(f'Negative Values: {(responses['revenue_generated'] < 0).sum()}')

Responses Data Overview:
------------------------------------
Responses Shape: (15000, 8)
------------------------------------
Responses Info:
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   reponses_id        15000 non-null  int64  
 1   campaign_id        15000 non-null  int64  
 2   customer_id        15000 non-null  int64  
 3   sent               15000 non-null  int64  
 4   opended            15000 non-null  int64  
 5   clicked            15000 non-null  int64  
 6   converted          15000 non-null  int64  
 7   revenue_generated  15000 non-null  float64
dtypes: float64(1), int64(7)
memory usage: 937.6 KB
None
------------------------------------
Responses Describe:
        reponses_id   campaign_id   customer_id     sent       opended  \
count  15000.000000  15000.000000  15000.000000  15000.0  15000.000000   
mean    7500.50000

In [10]:
# Cleaning Data

# Removing Duplicates:
orders = orders.drop_duplicates(subset='order_id', keep='first')

# Drop Null Values:
orders = orders.dropna(subset=['amount'])

# Remove Negative Values:
orders = orders[orders['amount'] >= 0]

In [13]:
# Data Preparation for RFM Analysis

# Keeping only orders with status 'Completed'
orders_completed = orders[orders['status'] == 'Completed'].copy()

# Parse Dates
orders_completed['order_date'] = pd.to_datetime(orders_completed['order_date'])
customers['signup_date'] = pd.to_datetime(customers['signup_date'])

# Remove orders with invalid customer_ids
valid_ids = set(customers['customer_id'])
orders_completed = orders_completed[orders_completed['customer_id'].isin(valid_ids)]


In [14]:
# Cleaning Campaign Responses

# Dropping Duplicates:
responses = responses.drop_duplicates(subset='reponses_id', keep='first')

# Ensuring clicked=1 only if opened=1:
responses.loc[responses['opended'] == 0, 'clicked'] = 0

# Ensuring converted=1 only if clicked=1:
responses.loc[responses['clicked'] == 0, 'converted'] = 0

# Null Revenue for non-converted responses:
responses.loc[responses['converted'] == 0, 'revenue_generated'] = 0

In [15]:
orders.head()

,order_id,customer_id,order_date,amount,category,channel,status
150,5151,4023,2024-09-01,421.39,Apparel,Organic,Returned
151,5152,2394,2024-12-09,114.06,Electronics,Email,Cancelled
152,5153,2698,2024-11-09,472.13,Sports,Google Search,Completed
153,5154,4151,2022-10-21,243.56,Beauty,Email,Completed
154,5155,5511,2023-04-18,346.77,Sports,Google Search,Cancelled


In [16]:
customers.head()

,customer_id,name,email,city,signup_channel,signup_date,age,gender
0,1001,customer_0,customer0.gmail.com,Delhi,Organic,2023-05-07,52,M
1,1002,customer_1,customer1.gmail.com,Delhi,SMS,2024-06-03,36,F
2,1003,customer_2,customer2.gmail.com,Bengaluru,SMS,2022-02-25,48,M
3,1004,customer_3,customer3.gmail.com,Chennai,SMS,2023-05-31,37,F
4,1005,customer_4,customer4.gmail.com,Mumbai,Google Search,2022-01-05,38,M


In [17]:
responses.head()

,reponses_id,campaign_id,customer_id,sent,opended,clicked,converted,revenue_generated
0,1,5,2071,1,1,1,0,0.0
1,2,1,4770,1,1,0,0,0.0
2,3,5,1294,1,1,0,0,0.0
3,4,7,2750,1,1,0,0,0.0
4,5,1,5432,1,1,1,0,0.0


In [18]:
responses[responses['revenue_generated'] > 0]

,reponses_id,campaign_id,customer_id,sent,opended,clicked,converted,revenue_generated
597,598,1,3313,1,1,1,1,236.33
626,627,6,3868,1,1,1,1,155.15
730,731,2,1458,1,1,1,1,117.98
871,872,6,5391,1,1,1,1,399.93
971,972,5,4727,1,1,1,1,66.81
...,...,...,...,...,...,...,...,...
14430,14431,5,1457,1,1,1,1,96.93
14569,14570,6,4639,1,1,1,1,292.84
14574,14575,5,3742,1,1,1,1,938.43
14590,14591,5,4820,1,1,1,1,177.97


In [ ]:
# Saving Clean Files
'''
orders_completed.to_csv('clean_orders.csv', index=False)
customers.to_csv('clean_customers.csv', index=False)
responses.to_csv('clean_responses.csv', index=False)
'''